# Customer Churn Prediction Using PySpark

## CRISP-DM Data Science Project

This project uses the UCI Iranian Churn Dataset to understand customer
churn and build a machine learning model using PySpark.

## 1. Business Understanding

### Business Problem

A telecommunications company wants to understand customer churn and
identify factors associated with customers leaving the service.

Customer churn can reduce revenue and increase the cost of acquiring
replacement customers. Understanding churn patterns can help the
business develop better customer-retention strategies.

### Project Objectives

- Understand the characteristics of customers who churn.
- Identify variables associated with churn.
- Explore differences between churned and non-churned customers.
- Prepare the data for machine learning.
- Build a churn prediction model using PySpark ML.
- Evaluate the predictive performance of the model.
- Provide business recommendations based on the findings.

### Business Questions

1. What proportion of customers churn?
2. Which customer characteristics are associated with churn?
3. Do complaints relate to customer churn?
4. Does subscription length differ between churned and non-churned customers?
5. How does customer value differ between the two groups?
6. Can we predict customer churn using customer behavior and service information?

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (col,count,when,avg,sum,min,max,round,desc)
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import (BinaryClassificationEvaluator,MulticlassClassificationEvaluator)

In [2]:
spark = (
    SparkSession.builder
    .appName("Iranian Customer Churn Prediction")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

In [3]:
spark

## 2. Data Understanding

In [4]:
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [6]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
iranian_churn = fetch_ucirepo(id=563) 
  
# data (as pandas dataframes) 
X = iranian_churn.data.features 
y = iranian_churn.data.targets 
  
# metadata 
print(iranian_churn.metadata) 
  
# variable information 
print(iranian_churn.variables) 


{'uci_id': 563, 'name': 'Iranian Churn', 'repository_url': 'https://archive.ics.uci.edu/dataset/563/iranian+churn+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/563/data.csv', 'abstract': "This dataset is randomly collected from an Iranian telecom company's database over a period of 12 months.", 'area': 'Business', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 3150, 'num_features': 13, 'feature_types': ['Integer'], 'demographics': ['Age'], 'target_col': ['Churn'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2020, 'last_updated': 'Sat Mar 09 2024', 'dataset_doi': '10.24432/C5JW3Z', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': 'This dataset is randomly collected from an Iranian telecom companyâ€™s database over a period of 12 months. A total of 3150 rows of data, each representing a customer, bear information for 13 columns. The attribu

In [9]:
import pandas as pd

data_pd = pd.concat([X, y], axis=1)

data_pd.head()

,Call Failure,Complains,Subscription Length,Charge Amount,Seconds of Use,Frequency of use,Frequency of SMS,Distinct Called Numbers,Age Group,Tariff Plan,Status,Age,Customer Value,Churn
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0
1,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0


In [10]:
# check dimensions
print("Rows:", data_pd.shape[0])
print("Columns:", data_pd.shape[1])

Rows: 3150
Columns: 14


In [13]:
# convert to a Spark DataFrame
df = spark.createDataFrame(data_pd)
df.show(5)

+-------------+---------+--------------------+--------------+--------------+----------------+----------------+-----------------------+---------+-----------+------+---+--------------+-----+
|Call  Failure|Complains|Subscription  Length|Charge  Amount|Seconds of Use|Frequency of use|Frequency of SMS|Distinct Called Numbers|Age Group|Tariff Plan|Status|Age|Customer Value|Churn|
+-------------+---------+--------------------+--------------+--------------+----------------+----------------+-----------------------+---------+-----------+------+---+--------------+-----+
|            8|        0|                  38|             0|          4370|              71|               5|                     17|        3|          1|     1| 30|        197.64|    0|
|            0|        0|                  39|             0|           318|               5|               7|                      4|        2|          1|     2| 25|        46.035|    0|
|           10|        0|                  37|         

In [14]:
df.printSchema()

root
 |-- Call  Failure: long (nullable = true)
 |-- Complains: long (nullable = true)
 |-- Subscription  Length: long (nullable = true)
 |-- Charge  Amount: long (nullable = true)
 |-- Seconds of Use: long (nullable = true)
 |-- Frequency of use: long (nullable = true)
 |-- Frequency of SMS: long (nullable = true)
 |-- Distinct Called Numbers: long (nullable = true)
 |-- Age Group: long (nullable = true)
 |-- Tariff Plan: long (nullable = true)
 |-- Status: long (nullable = true)
 |-- Age: long (nullable = true)
 |-- Customer Value: double (nullable = true)
 |-- Churn: long (nullable = true)



In [15]:
# summary statistics
df.describe().show()

+-------+-----------------+-------------------+--------------------+------------------+-----------------+-----------------+------------------+-----------------------+------------------+------------------+-------------------+------------------+-----------------+-------------------+
|summary|    Call  Failure|          Complains|Subscription  Length|    Charge  Amount|   Seconds of Use| Frequency of use|  Frequency of SMS|Distinct Called Numbers|         Age Group|       Tariff Plan|             Status|               Age|   Customer Value|              Churn|
+-------+-----------------+-------------------+--------------------+------------------+-----------------+-----------------+------------------+-----------------------+------------------+------------------+-------------------+------------------+-----------------+-------------------+
|  count|             3150|               3150|                3150|              3150|             3150|             3150|              3150|            

In [16]:
print(iranian_churn.variables)

                       name     role        type demographic description  \
0             Call  Failure  Feature     Integer        None        None   
1                 Complains  Feature      Binary        None        None   
2      Subscription  Length  Feature     Integer        None        None   
3            Charge  Amount  Feature     Integer        None        None   
4            Seconds of Use  Feature     Integer        None        None   
5          Frequency of use  Feature     Integer        None        None   
6          Frequency of SMS  Feature     Integer        None        None   
7   Distinct Called Numbers  Feature     Integer        None        None   
8                 Age Group  Feature     Integer         Age        None   
9               Tariff Plan  Feature     Integer        None        None   
10                   Status  Feature      Binary        None        None   
11                      Age  Feature     Integer         Age        None   
12          

In [17]:
df.show(10, truncate=False)

+-------------+---------+--------------------+--------------+--------------+----------------+----------------+-----------------------+---------+-----------+------+---+--------------+-----+
|Call  Failure|Complains|Subscription  Length|Charge  Amount|Seconds of Use|Frequency of use|Frequency of SMS|Distinct Called Numbers|Age Group|Tariff Plan|Status|Age|Customer Value|Churn|
+-------------+---------+--------------------+--------------+--------------+----------------+----------------+-----------------------+---------+-----------+------+---+--------------+-----+
|8            |0        |38                  |0             |4370          |71              |5               |17                     |3        |1          |1     |30 |197.64        |0    |
|0            |0        |39                  |0             |318           |5               |7               |4                      |2        |1          |2     |25 |46.035        |0    |
|10           |0        |37                  |0        

In [18]:
# examine target variable
df.groupBy("Churn").count().orderBy("Churn").show()

+-----+-----+
|Churn|count|
+-----+-----+
|    0| 2655|
|    1|  495|
+-----+-----+



In [19]:
# calculate percentage
total_customers = df.count()

(
    df.groupBy("Churn")
      .count()
      .withColumn(
          "percentage",
          round(col("count") / total_customers * 100, 2)
      )
      .orderBy("Churn")
      .show()
)

+-----+-----+----------+
|Churn|count|percentage|
+-----+-----+----------+
|    0| 2655|     84.29|
|    1|  495|     15.71|
+-----+-----+----------+



In [20]:
# Check Missing values
missing_values = df.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

missing_values.show()

+-------------+---------+--------------------+--------------+--------------+----------------+----------------+-----------------------+---------+-----------+------+---+--------------+-----+
|Call  Failure|Complains|Subscription  Length|Charge  Amount|Seconds of Use|Frequency of use|Frequency of SMS|Distinct Called Numbers|Age Group|Tariff Plan|Status|Age|Customer Value|Churn|
+-------------+---------+--------------------+--------------+--------------+----------------+----------------+-----------------------+---------+-----------+------+---+--------------+-----+
|            0|        0|                   0|             0|             0|               0|               0|                      0|        0|          0|     0|  0|             0|    0|
+-------------+---------+--------------------+--------------+--------------+----------------+----------------+-----------------------+---------+-----------+------+---+--------------+-----+



In [21]:
# Check duplicates
total_rows = df.count()
distinct_rows = df.distinct().count()

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Duplicate rows:", total_rows - distinct_rows)

Total rows: 3150
Distinct rows: 2850
Duplicate rows: 300


In [23]:
# churn by complaints
(
    df.groupBy("Complains", "Churn")
      .count()
      .orderBy("Complains", "Churn")
      .show()
)

+---------+-----+-----+
|Complains|Churn|count|
+---------+-----+-----+
|        0|    0| 2614|
|        0|    1|  295|
|        1|    0|   41|
|        1|    1|  200|
+---------+-----+-----+



In [24]:
# churn by tariff plan
(
    df.groupBy("Tariff Plan", "Churn")
      .count()
      .orderBy("Tariff Plan", "Churn")
      .show()
)

+-----------+-----+-----+
|Tariff Plan|Churn|count|
+-----------+-----+-----+
|          1|    0| 2416|
|          1|    1|  489|
|          2|    0|  239|
|          2|    1|    6|
+-----------+-----+-----+



In [25]:
# churn by age group
(
    df.groupBy("Age Group", "Churn")
      .count()
      .orderBy("Age Group", "Churn")
      .show()
)

+---------+-----+-----+
|Age Group|Churn|count|
+---------+-----+-----+
|        1|    0|  123|
|        2|    0|  853|
|        2|    1|  184|
|        3|    0| 1195|
|        3|    1|  230|
|        4|    0|  316|
|        4|    1|   79|
|        5|    0|  168|
|        5|    1|    2|
+---------+-----+-----+



##  3.Data Preparation

In [28]:
# Data Cleaning
df_clean = df.dropDuplicates() # drop duplicates

In [29]:
print("Original rows:", df.count())
print("Rows after removing duplicates:", df_clean.count())

Original rows: 3150
Rows after removing duplicates: 2850


In [30]:
# check missing values
missing_values = df_clean.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_clean.columns
])

missing_values.show()

+-------------+---------+--------------------+--------------+--------------+----------------+----------------+-----------------------+---------+-----------+------+---+--------------+-----+
|Call  Failure|Complains|Subscription  Length|Charge  Amount|Seconds of Use|Frequency of use|Frequency of SMS|Distinct Called Numbers|Age Group|Tariff Plan|Status|Age|Customer Value|Churn|
+-------------+---------+--------------------+--------------+--------------+----------------+----------------+-----------------------+---------+-----------+------+---+--------------+-----+
|            0|        0|                   0|             0|             0|               0|               0|                      0|        0|          0|     0|  0|             0|    0|
+-------------+---------+--------------------+--------------+--------------+----------------+----------------+-----------------------+---------+-----------+------+---+--------------+-----+



In [32]:
# check data types
df_clean.printSchema()

root
 |-- Call  Failure: long (nullable = true)
 |-- Complains: long (nullable = true)
 |-- Subscription  Length: long (nullable = true)
 |-- Charge  Amount: long (nullable = true)
 |-- Seconds of Use: long (nullable = true)
 |-- Frequency of use: long (nullable = true)
 |-- Frequency of SMS: long (nullable = true)
 |-- Distinct Called Numbers: long (nullable = true)
 |-- Age Group: long (nullable = true)
 |-- Tariff Plan: long (nullable = true)
 |-- Status: long (nullable = true)
 |-- Age: long (nullable = true)
 |-- Customer Value: double (nullable = true)
 |-- Churn: long (nullable = true)



In [33]:
# check target variable
df_clean.groupBy("Churn").count().orderBy("Churn").show()

+-----+-----+
|Churn|count|
+-----+-----+
|    0| 2404|
|    1|  446|
+-----+-----+



In [38]:
df_clean = df_clean.toDF(
    *[column.replace("  ", "_").replace(" ", "_") for column in df_clean.columns]
)

df_clean.columns

['Call_Failure',
 'Complains',
 'Subscription_Length',
 'Charge_Amount',
 'Seconds_of_Use',
 'Frequency_of_use',
 'Frequency_of_SMS',
 'Distinct_Called_Numbers',
 'Age_Group',
 'Tariff_Plan',
 'Status',
 'Age',
 'Customer_Value',
 'Churn']

In [39]:
# select predictor variables
feature_columns = [
    "Call_Failure",
    "Complains",
    "Subscription_Length",
    "Charge_Amount",
    "Seconds_of_Use",
    "Frequency_of_use",
    "Frequency_of_SMS",
    "Distinct_Called_Numbers",
    "Age_Group",
    "Tariff_Plan",
    "Status",
    "Customer_Value"
]

In [40]:
df_clean.select(feature_columns + ["Churn"]).show(5)

+------------+---------+-------------------+-------------+--------------+----------------+----------------+-----------------------+---------+-----------+------+--------------+-----+
|Call_Failure|Complains|Subscription_Length|Charge_Amount|Seconds_of_Use|Frequency_of_use|Frequency_of_SMS|Distinct_Called_Numbers|Age_Group|Tariff_Plan|Status|Customer_Value|Churn|
+------------+---------+-------------------+-------------+--------------+----------------+----------------+-----------------------+---------+-----------+------+--------------+-----+
|          12|        0|                 37|            1|          3050|              44|              14|                     18|        3|          1|     1|        179.76|    1|
|           5|        0|                 35|            0|          6658|              72|             103|                     35|        2|          1|     1|        766.35|    0|
|           8|        0|                 43|            2|          6760|              96|

In [41]:
# Assemble the features
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

In [42]:
df_prepared = assembler.transform(df_clean)

In [43]:
df_prepared.select(
    "features",
    "Churn"
).show(5, truncate=False)

+----------------------------------------------------------------+-----+
|features                                                        |Churn|
+----------------------------------------------------------------+-----+
|[12.0,0.0,37.0,1.0,3050.0,44.0,14.0,18.0,3.0,1.0,1.0,179.76]    |1    |
|[5.0,0.0,35.0,0.0,6658.0,72.0,103.0,35.0,2.0,1.0,1.0,766.35]    |0    |
|[8.0,0.0,43.0,2.0,6760.0,96.0,201.0,32.0,3.0,1.0,1.0,1078.24]   |0    |
|[8.0,0.0,33.0,0.0,15405.0,180.0,140.0,26.0,2.0,1.0,1.0,1331.325]|0    |
|[8.0,0.0,37.0,1.0,6718.0,75.0,108.0,37.0,2.0,1.0,1.0,791.685]   |0    |
+----------------------------------------------------------------+-----+
only showing top 5 rows


In [45]:
# prepare modelling dataset
model_df = df_prepared.select("features","Churn")
model_df.show(5, truncate=False)

+----------------------------------------------------------------+-----+
|features                                                        |Churn|
+----------------------------------------------------------------+-----+
|[12.0,0.0,37.0,1.0,3050.0,44.0,14.0,18.0,3.0,1.0,1.0,179.76]    |1    |
|[5.0,0.0,35.0,0.0,6658.0,72.0,103.0,35.0,2.0,1.0,1.0,766.35]    |0    |
|[8.0,0.0,43.0,2.0,6760.0,96.0,201.0,32.0,3.0,1.0,1.0,1078.24]   |0    |
|[8.0,0.0,33.0,0.0,15405.0,180.0,140.0,26.0,2.0,1.0,1.0,1331.325]|0    |
|[8.0,0.0,37.0,1.0,6718.0,75.0,108.0,37.0,2.0,1.0,1.0,791.685]   |0    |
+----------------------------------------------------------------+-----+
only showing top 5 rows


##### Train-Test Split
The prepared dataset is divided into training and testing sets.
The training data will be used to fit the model, while the test data
will be used to evaluate performance on unseen observations.

In [46]:
# Train-Test Split
train_df, test_df = model_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Training rows: 2330
Testing rows: 520


In [47]:
train_df.printSchema()

root
 |-- features: vector (nullable = true)
 |-- Churn: long (nullable = true)



In [48]:
train_df.groupBy("Churn").count().show()
test_df.groupBy("Churn").count().show()

+-----+-----+
|Churn|count|
+-----+-----+
|    0| 1970|
|    1|  360|
+-----+-----+

+-----+-----+
|Churn|count|
+-----+-----+
|    0|  434|
|    1|   86|
+-----+-----+



## 4. Modeling

Logistic Regression is used to predict whether a customer is likely to
churn based on the selected customer characteristics.

In [49]:
from pyspark.ml.classification import LogisticRegression

In [50]:
# create a model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="Churn",
    maxIter=100
)

In [51]:
# Train model
lr_model = lr.fit(train_df)

In [52]:
# Generate predictions
predictions = lr_model.transform(test_df)

In [53]:
predictions.select(
    "Churn",
    "prediction",
    "probability"
).show(10, truncate=False)

+-----+----------+-----------------------------------------+
|Churn|prediction|probability                              |
+-----+----------+-----------------------------------------+
|1    |1.0       |[0.013126565340368402,0.9868734346596316]|
|1    |1.0       |[0.016786866699726934,0.983213133300273] |
|1    |1.0       |[0.02144565161270756,0.9785543483872925] |
|0    |0.0       |[0.8592018546714164,0.1407981453285836]  |
|0    |0.0       |[0.5134895690038085,0.4865104309961915]  |
|0    |0.0       |[0.8106205209951861,0.1893794790048139]  |
|1    |0.0       |[0.6065625374105372,0.39343746258946277] |
|1    |0.0       |[0.6842179874198258,0.31578201258017424] |
|0    |0.0       |[0.5576176484564255,0.4423823515435745]  |
|0    |1.0       |[0.4603859552387666,0.5396140447612334]  |
+-----+----------+-----------------------------------------+
only showing top 10 rows


In [54]:
# Examine the model coefficients
coefficients = lr_model.coefficients

for feature, coefficient in zip(feature_columns, coefficients):
    print(f"{feature}: {coefficient:.4f}")

Call_Failure: 0.1405
Complains: 4.2974
Subscription_Length: -0.0273
Charge_Amount: -0.5118
Seconds_of_Use: 0.0001
Frequency_of_use: -0.0591
Frequency_of_SMS: -0.0480
Distinct_Called_Numbers: -0.0006
Age_Group: 0.1951
Tariff_Plan: 0.7593
Status: 1.3509
Customer_Value: 0.0085


In [55]:
# Generate a confusion matrix
(
    predictions
    .groupBy("Churn", "prediction")
    .count()
    .orderBy("Churn", "prediction")
    .show()
)

+-----+----------+-----+
|Churn|prediction|count|
+-----+----------+-----+
|    0|       0.0|  420|
|    0|       1.0|   14|
|    1|       0.0|   46|
|    1|       1.0|   40|
+-----+----------+-----+



##### Modeling Summary

A Logistic Regression classifier was trained using the prepared
customer features.

The model was fitted on the training dataset and used to generate
churn predictions for the unseen test dataset.

Model coefficients and a confusion matrix were examined to understand
the model's behavior before formal evaluation.

## 5. Evaluation

The model is evaluated using accuracy, precision, recall, F1-score,
and ROC-AUC.

These metrics provide different perspectives on classification
performance, which is important because correctly identifying customers
who are likely to churn is a key business objective.

In [57]:
from pyspark.ml.evaluation import (MulticlassClassificationEvaluator,BinaryClassificationEvaluator)

In [58]:
# Accuracy
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="Churn",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = accuracy_evaluator.evaluate(predictions)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.8846


In [59]:
# Precision
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="Churn",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

precision = precision_evaluator.evaluate(predictions)

print(f"Precision: {precision:.4f}")

Precision: 0.8747


In [60]:
# Recall
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="Churn",
    predictionCol="prediction",
    metricName="weightedRecall"
)

recall = recall_evaluator.evaluate(predictions)

print(f"Recall: {recall:.4f}")

Recall: 0.8846


In [61]:
# F1-Score
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="Churn",
    predictionCol="prediction",
    metricName="f1"
)

f1 = f1_evaluator.evaluate(predictions)

print(f"F1 Score: {f1:.4f}")

F1 Score: 0.8735


In [62]:
# ROC-AUC
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="Churn",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = auc_evaluator.evaluate(predictions)

print(f"ROC-AUC: {auc:.4f}")

ROC-AUC: 0.9106


In [63]:
# metrics
metrics = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "ROC-AUC": auc
}

for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

Accuracy: 0.8846
Precision: 0.8747
Recall: 0.8846
F1 Score: 0.8735
ROC-AUC: 0.9106
